# Multi-Modal Restaurant Recommendation System

A walkthrough of a recommender that fuses **review text** (DistilBERT), **food images**
(ResNet18) and **structured attributes** (cuisine / price / location) into a unified
hybrid ranking model — combining collaborative + content-based filtering with
cold-start handling. Mirrors the kind of discovery feed Zomato/Yelp run.

**Pipeline:** synthetic multi-modal data → frozen encoders → late fusion → 
matrix-factorization CF + content profiles → unified MLP ranker → cold-start regressors.

> Run `python -m restaurant_rec.cli all` once first so the cached embeddings and trained
> model artifacts exist; the cells below load them.

## 1. The data
A coherent synthetic world: hidden factors (taste, quality, service, presentation) drive
ratings, and each modality surfaces a different slice of that signal.

In [ ]:
from restaurant_rec.data.generate import load_dataset, summarize
data = load_dataset()
print(summarize(data))

In [ ]:
data['restaurants'].head(3)

In [ ]:
# a few generated listing reviews
print(data['reviews']['text'].head(4).to_string(index=False))

In [ ]:
# show a generated food image
import matplotlib.pyplot as plt
from matplotlib.image import imread
from restaurant_rec.config import DATA_DIR
row = data['restaurants'].iloc[0]
img = imread(DATA_DIR / row['image_path'])
plt.figure(figsize=(3,3)); plt.imshow(img); plt.axis('off')
plt.title(f"{row['name']}\n({row['cuisine']})"); plt.show()

## 2. Frozen encoders → modality embeddings
DistilBERT (768-d, mean-pooled), ResNet18 (512-d), and a structured encoder. These are
cached to `artifacts/` by `build-embeddings`.

In [ ]:
from restaurant_rec.pipeline import get_modality_matrices
from restaurant_rec.config import CONFIG
mats = get_modality_matrices(data, CONFIG)
{k: v.shape for k, v in mats.items()}

In [ ]:
# Do the ResNet image features cluster by cuisine? (PCA to 2-D)
import numpy as np
from sklearn.decomposition import PCA
from restaurant_rec.data import schema as S
rest = data['restaurants'].sort_values(S.REST_ID)
emb2 = PCA(n_components=2, random_state=0).fit_transform(mats['image'])
cuis = rest['cuisine'].to_numpy()
plt.figure(figsize=(7,5))
for c in sorted(set(cuis))[:6]:
    m = cuis == c
    plt.scatter(emb2[m,0], emb2[m,1], s=12, label=c)
plt.legend(fontsize=8, ncol=2); plt.title('ResNet image features by cuisine (PCA)')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

## 3. Train (or load) the models
Late fusion stats, matrix-factorization CF, the unified ranker, and the cold-start
regressors. Skip if artifacts already exist.

In [ ]:
from restaurant_rec.config import RANKER_MODEL_PT
if not RANKER_MODEL_PT.exists():
    from restaurant_rec.train import train_all
    train_all(CONFIG)
else:
    print('artifacts already present — skipping training')

## 4. Evaluation — does fusion help?
Compare the hybrid ranker against CF-only and content-only on held-out interactions.

In [ ]:
from restaurant_rec.pipeline import assemble_state
from restaurant_rec.evaluate import evaluate_models, format_table
state = assemble_state(CONFIG)
res = evaluate_models(state, CONFIG)
k = CONFIG.model.eval_k
print(format_table('Model comparison', res, [f'P@{k}', f'R@{k}', f'NDCG@{k}', f'MAP@{k}', 'coverage']))

In [ ]:
# NDCG@K bar chart
names = list(res); ndcg = [res[n][f'NDCG@{k}'] for n in names]
plt.figure(figsize=(5,3)); plt.bar(names, ndcg, color=['#2a9d8f','#e9c46a','#e76f51'])
plt.ylabel(f'NDCG@{k}'); plt.title('Hybrid vs single-paradigm'); plt.xticks(rotation=15); plt.show()

## 5. Modality ablations
Retrain the ranker dropping one modality at a time — each one should add a little lift.

In [ ]:
from restaurant_rec.evaluate import evaluate_ablations
abl = evaluate_ablations(CONFIG)
print(format_table('Ablations', abl, [f'NDCG@{k}', f'MAP@{k}', 'coverage']))

In [ ]:
names = list(abl); ndcg = [abl[n][f'NDCG@{k}'] for n in names]
plt.figure(figsize=(7,3)); plt.bar(names, ndcg, color='#264653')
plt.ylabel(f'NDCG@{k}'); plt.title('Modality ablations'); plt.xticks(rotation=25, ha='right'); plt.show()

## 6. Cold-start
New users (bootstrapped from onboarding preferences) and new restaurants
(CF vector predicted from content).

In [ ]:
from restaurant_rec.evaluate import evaluate_coldstart
cold = evaluate_coldstart(state, CONFIG)
print(format_table('Cold-start', cold, [f'P@{k}', f'R@{k}', f'NDCG@{k}', 'coverage']))

In [ ]:
# Recommendations for an existing user (note the learned taste consistency)
from restaurant_rec.recommend import recommend_for_user, recommend_cold_user
recommend_for_user(5, k=8, state=state)

In [ ]:
# Cold-start: a brand-new user described only by preferences
recommend_cold_user(['Italian','Japanese'], price=2, city='Bayport', k=8, state=state)

## Takeaways
- **Fusion wins:** the hybrid ranker beats CF-only and content-only on NDCG/MAP.
- **Each modality contributes:** ablations degrade as modalities are removed.
- **Cold users** are served well from onboarding preferences; **cold items** land mid-pack
  (the classic exposure problem) rather than being dropped.
- Swap the synthetic data for real reviews/photos by replacing the CSVs and images in
  `data/` — no model code changes needed.